# 05.1 Capstone: Tabular Classification

This notebook is a capstone exercise that follows the rhythm of a small real project. The goal is to define the problem, process data, run a classical baseline, train PyTorch models, compare controlled experiments, and explain the results.

The important habit is comparison. A neural network is not automatically better than a simple baseline. The project should show what was compared, how it was evaluated, and what conclusion is supported by the evidence.

## Learning Goals

After this notebook, you should be able to:

1. Run a tabular classification task end to end.
2. Compare a baseline model with an `MLP`.
3. Organize train, validation, and test workflows.
4. Build a comparison table for controlled experiments.
5. Analyze results with a confusion matrix and misclassified samples.
6. Write a conclusion that looks like a real project summary.

In [ ]:
import copy
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(42)

## Problem Definition

We use the breast cancer dataset for binary classification. Each sample has numeric tabular features, and the target is the tumor class. This dataset is small enough for teaching, has no image or text preprocessing, and is a good setting for comparing a simple baseline with a small neural network.

The modeling question is not just "can we train an MLP?" It is "does the MLP add value compared with a stable tabular baseline?"

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
df.rename(columns={"target": "label"}, inplace=True)

print("dataset shape =", df.shape)
print("label counts =\n", df["label"].value_counts())
print("target names =", list(data.target_names))
df.head(3)

## Data Split and Standardization

A very common preprocessing step for tabular data is `standardization`.

We follow this order strictly:

1. split into train
2. fit the scaler only on the training split
3. apply the same scaler to validation and test

In [ ]:
X = df.drop(columns=["label"])
y = df["label"]
feature_names = list(X.columns)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("train shape =", X_train_scaled.shape)
print("val shape =", X_val_scaled.shape)
print("test shape =", X_test_scaled.shape)
print("train positive rate / train positive rate =", y_train.mean())

## 3. Baseline: Logistic Regression

`Logistic Regression` is a very common baseline for tabular classification.

Its role is not to be state of the art, but to provide a clear, reliable, and interpretable reference point.


In [ ]:
baseline_model = LogisticRegression(max_iter=3000, random_state=42)
baseline_model.fit(X_train_scaled, y_train)

baseline_val_preds = baseline_model.predict(X_val_scaled)
baseline_test_preds = baseline_model.predict(X_test_scaled)

baseline_val_acc = accuracy_score(y_val, baseline_val_preds)
baseline_test_acc = accuracy_score(y_test, baseline_test_preds)

print("baseline val acc =", round(baseline_val_acc, 4))
print("baseline test acc =", round(baseline_test_acc, 4))

## Build the PyTorch Data Pipeline

Next we move to the `PyTorch` experiments.

We continue using the already standardized numeric features.


In [ ]:
train_ds = TensorDataset(
    torch.tensor(X_train_scaled, dtype=torch.float32),
    torch.tensor(y_train.values, dtype=torch.long),
)
val_ds = TensorDataset(
    torch.tensor(X_val_scaled, dtype=torch.float32),
    torch.tensor(y_val.values, dtype=torch.long),
)
test_ds = TensorDataset(
    torch.tensor(X_test_scaled, dtype=torch.float32),
    torch.tensor(y_test.values, dtype=torch.long),
)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

xb, yb = next(iter(train_loader))
print("xb.shape =", xb.shape)
print("yb.shape =", yb.shape)

## Model, Training Function, and Evaluation Function

To keep the comparison fair, we let different `MLP` variants share the same training logic.


In [ ]:
class TabularMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=32, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x):
        return self.net(x)


def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_items = 0

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            preds = logits.argmax(dim=1)
            total_loss += loss.item() * xb.size(0)
            total_correct += (preds == yb).sum().item()
            total_items += xb.size(0)

    return total_loss / total_items, total_correct / total_items


def train_torch_model(config):
    set_seed(42)
    model = TabularMLP(
        in_dim=X_train_scaled.shape[1],
        hidden_dim=config["hidden_dim"],
        dropout=config["dropout"],
    )
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    history = []
    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = -1.0

    for epoch in range(1, config["epochs"] + 1):
        train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
            }
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


def collect_predictions(model, loader):
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for xb, yb in loader:
            preds = model(xb).argmax(dim=1)
            all_preds.append(preds)
            all_targets.append(yb)
    return torch.cat(all_preds), torch.cat(all_targets)

## Controlled Experiments

We run two MLP experiments so the comparison has a clear purpose. One model is smaller. The other has a larger hidden layer plus dropout and weight decay. This lets us ask whether extra capacity and regularization improve validation and test performance.

Only change a small number of variables at a time; otherwise it becomes difficult to explain why results changed.

In [ ]:
experiment_configs = {
    "MLP-Small": {
        "hidden_dim": 16,
        "dropout": 0.0,
        "lr": 0.01,
        "weight_decay": 0.0,
        "epochs": 25,
    },
    "MLP-Regularized": {
        "hidden_dim": 64,
        "dropout": 0.2,
        "lr": 0.01,
        "weight_decay": 1e-4,
        "epochs": 30,
    },
}

torch_runs = {}
for name, cfg in experiment_configs.items():
    model, history_df = train_torch_model(cfg)
    test_preds, test_targets = collect_predictions(model, test_loader)
    val_preds, val_targets = collect_predictions(model, val_loader)
    torch_runs[name] = {
        "config": cfg,
        "model": model,
        "history": history_df,
        "val_acc": accuracy_score(val_targets.numpy(), val_preds.numpy()),
        "test_acc": accuracy_score(test_targets.numpy(), test_preds.numpy()),
        "test_preds": test_preds.numpy(),
        "test_targets": test_targets.numpy(),
    }
    print(name, "best val acc =", round(torch_runs[name]["val_acc"], 4), "| test acc =", round(torch_runs[name]["test_acc"], 4))

## Result Table

A project should at minimum have a clear comparison table.


In [ ]:
results_df = pd.DataFrame(
    [
        {
            "model": "LogisticRegression",
            "category": "baseline",
            "val_acc": round(baseline_val_acc, 4),
            "test_acc": round(baseline_test_acc, 4),
            "notes": "classic tabular baseline",
        },
        {
            "model": "MLP-Small",
            "category": "torch",
            "val_acc": round(torch_runs["MLP-Small"]["val_acc"], 4),
            "test_acc": round(torch_runs["MLP-Small"]["test_acc"], 4),
            "notes": "smaller hidden layer",
        },
        {
            "model": "MLP-Regularized",
            "category": "torch",
            "val_acc": round(torch_runs["MLP-Regularized"]["val_acc"], 4),
            "test_acc": round(torch_runs["MLP-Regularized"]["test_acc"], 4),
            "notes": "larger hidden layer + dropout + weight decay",
        },
    ]
).sort_values(by=["test_acc", "val_acc"], ascending=False)
results_df

## Training Curves

We only plot curves for the `PyTorch` models here because the baseline does not have epoch-by-epoch history.


In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
for name, run in torch_runs.items():
    hist = run["history"]
    plt.plot(hist["epoch"], hist["train_loss"], label=f"{name} train")
    plt.plot(hist["epoch"], hist["val_loss"], linestyle="--", label=f"{name} val")
plt.title("Loss Curves")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend(fontsize=8)

plt.subplot(1, 2, 2)
for name, run in torch_runs.items():
    hist = run["history"]
    plt.plot(hist["epoch"], hist["train_acc"], label=f"{name} train")
    plt.plot(hist["epoch"], hist["val_acc"], linestyle="--", label=f"{name} val")
plt.title("Accuracy Curves")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend(fontsize=8)

plt.tight_layout()
plt.show()
plt.close()

## Select the Best Model

Here we select the final model by the highest `test_acc`, only for teaching demonstration.

In real projects, you usually choose by validation performance and use the test set only once for final reporting.


In [ ]:
all_candidates = {
    "LogisticRegression": {
        "test_preds": baseline_test_preds,
        "test_targets": y_test.to_numpy(),
        "test_acc": baseline_test_acc,
    },
    "MLP-Small": {
        "test_preds": torch_runs["MLP-Small"]["test_preds"],
        "test_targets": torch_runs["MLP-Small"]["test_targets"],
        "test_acc": torch_runs["MLP-Small"]["test_acc"],
    },
    "MLP-Regularized": {
        "test_preds": torch_runs["MLP-Regularized"]["test_preds"],
        "test_targets": torch_runs["MLP-Regularized"]["test_targets"],
        "test_acc": torch_runs["MLP-Regularized"]["test_acc"],
    },
}

best_name = max(all_candidates, key=lambda name: all_candidates[name]["test_acc"])
best_preds = all_candidates[best_name]["test_preds"]
best_targets = all_candidates[best_name]["test_targets"]

print("best model =", best_name)
print("best test acc =", round(all_candidates[best_name]["test_acc"], 4))

## 10. Confusion Matrix

In binary classification, a `confusion matrix` helps you see:

- which class is easier to misclassify
- whether the main mistakes are `false positives` or `false negatives`

In [ ]:
cm = confusion_matrix(best_targets, best_preds)
plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap="Blues")
plt.title(f"Confusion Matrix: {best_name}")
plt.xlabel("predicted label")
plt.ylabel("true label")
plt.colorbar()

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

plt.tight_layout()
plt.show()
plt.close()

print(classification_report(best_targets, best_preds, digits=4, target_names=list(data.target_names)))

## Misclassified Sample Analysis

An important project step is: do not stop at overall metrics.

Below we inspect several misclassified samples and a few key features.


In [ ]:
test_index = y_test.index.to_numpy()
mis_positions = np.where(best_preds != best_targets)[0]
selected_positions = mis_positions[:8]
selected_columns = [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area",
    "mean smoothness",
]

if len(selected_positions) > 0:
    mis_df = X_test.iloc[selected_positions][selected_columns].copy()
    mis_df["true_label"] = y_test.iloc[selected_positions].to_numpy()
    mis_df["pred_label"] = best_preds[selected_positions]
    mis_df
else:
    print("No misclassified samples / no misclassified samples.")

## Final Conclusion

A practical project conclusion should at least answer:

1. which model is best in the end
2. whether there was improvement over the baseline
3. where the main errors are
4. what the next steps should be

In [ ]:
summary_lines = [
    f"Best model: {best_name}",
    f"Baseline test accuracy: {baseline_test_acc:.4f}",
    f"Best test accuracy: {all_candidates[best_name]['test_acc']:.4f}",
    f"Improvement over baseline: {all_candidates[best_name]['test_acc'] - baseline_test_acc:.4f}",
    f"Number of misclassified test samples: {int((best_preds != best_targets).sum())}",
]

for line in summary_lines:
    print(line)

In [ ]:
# Exercise 1
#
# If the MLP has very high train_acc but lower val_acc than the LogisticRegression
# baseline, what would you suspect first?
#
# Answer in full sentences. Your answer should mention overfitting and at least
# one practical next step, such as stronger regularization, a smaller model, or
# checking preprocessing.

Exercise 1 Reference Answer

I would first suspect `overfitting`.

Then I would consider stronger regularization, a smaller model, or better feature processing.


In [ ]:
# Exercise 2
#
# Explain why LogisticRegression is worth running first on many tabular tasks.
#
# Answer in full sentences. Your answer should mention that it is fast, stable,
# easy to interpret as a baseline, and often surprisingly strong on structured
# numeric data.

Exercise 2 Reference Answer

Because it is simple, stable, fast, and often already provides a strong baseline.

If you cannot beat the baseline, it often means the more complex model has not yet added real value.


## Summary

The most important outcome of this capstone is not a specific score, but the fact that you completed the full project workflow.

You have practiced:

1. problem definition
2. data splitting and standardization
3. comparing a baseline and improved models
4. result tables, training curves, and confusion matrices
5. misclassification analysis and conclusion writing